# 🧬 RecurrentBitNet V3: Selective BitLinear on Qwen3.5

**Hypothesis**: Gated DeltaNet layers are more robust to ternary (1.58-bit) quantization than standard attention layers, because their recurrent state matrix accumulates in full precision and the delta rule provides error-correcting memory updates.

**This notebook**:
1. Loads Qwen3.5-0.8B and measures baseline quality
2. Surgically converts 18 DeltaNet layers to BitLinear (ternary weights)
3. Measures quality degradation from quantization
4. Runs knowledge distillation to recover quality
5. Compares all three measurements

---

## 🔧 Setup & Configuration

In [ ]:
# Install dependencies
!pip install -q transformers>=4.51.0 datasets accelerate matplotlib

In [ ]:
# Clone the repo (Colab) or set path (local)
import os
import sys

REPO_URL = "https://github.com/YOUR_USERNAME/recurrent_bitnet.git"  # UPDATE THIS
REPO_DIR = "/content/recurrent_bitnet"

if os.path.exists("/content"):  # Running in Colab
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    sys.path.insert(0, REPO_DIR)
    # Mount Google Drive for checkpoints
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/recurrent_bitnet/checkpoints"
else:  # Running locally
    REPO_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    sys.path.insert(0, REPO_DIR)
    CHECKPOINT_DIR = os.path.join(REPO_DIR, "checkpoints")

print(f"Repo: {REPO_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
#@title Model Configuration { display-mode: "form" }
#@markdown Select your model size and GPU tier:

MODEL_SIZE = "0.8B"  #@param ["0.8B", "2B", "4B"]
SURGERY_AGGRESSION = "standard"  #@param ["conservative", "standard", "aggressive"]
NUM_DISTILL_STEPS = 5000  #@param {type: "integer"}

# Model name mapping
_MODEL_MAP = {
    "0.8B": "Qwen/Qwen3.5-0.8B-Base",
    "2B": "Qwen/Qwen3.5-2B-Base",
    "4B": "Qwen/Qwen3.5-4B-Base",
}
_TEACHER_MAP = {
    "0.8B": "Qwen/Qwen3.5-0.8B-Base",  # self-distillation
    "2B": "Qwen/Qwen3.5-4B-Base",
    "4B": "Qwen/Qwen3.5-9B-Base",
}

STUDENT_MODEL = _MODEL_MAP[MODEL_SIZE]
TEACHER_MODEL = _TEACHER_MAP[MODEL_SIZE]

# GPU detection
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_vram:.1f} GB)")
else:
    gpu_name = "CPU"
    gpu_vram = 0
    print("⚠️  No GPU detected. Training will be very slow.")

# Auto-select teacher based on VRAM
if gpu_vram < 20 and MODEL_SIZE != "0.8B":
    print(f"⚠️  {MODEL_SIZE} may not fit on {gpu_vram:.0f}GB. Consider 0.8B.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nStudent: {STUDENT_MODEL}")
print(f"Teacher: {TEACHER_MODEL}")
print(f"Surgery: {SURGERY_AGGRESSION}")
print(f"Steps:   {NUM_DISTILL_STEPS}")

In [ ]:
# Import RecurrentBitNet modules
from src.bitlinear import BitLinear, count_ternary_params
from src.surgery import convert_model, SurgeryConfig, surgical_report, identify_layer_types
from src.distill import (
    DistillationTrainer, DistillationConfig,
    create_dataloader, compute_model_perplexity,
)

print("✅ Modules imported successfully")

---
## 📊 Phase A: Baseline Measurement

Load the original Qwen3.5 model and measure its perplexity as our quality baseline.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model (FP32 for STE training)
print(f"Loading {STUDENT_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.float32,
    trust_remote_code=True,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded: {total_params:,} parameters")
print(f"Model size (FP32): {total_params * 4 / 1e9:.2f} GB")
print(f"Model size (FP16): {total_params * 2 / 1e9:.2f} GB")

In [ ]:
# Inspect the layer structure
layer_info = identify_layer_types(model)
print(f"{'Layer':>6} | {'Type':>10} | {'Class':>30} | Linears")
print("-" * 80)
for idx, info in layer_info.items():
    marker = "🔵" if info["type"] == "deltanet" else "🔴"
    linears = ", ".join(info["linears"][:4])  # first 4
    if len(info["linears"]) > 4:
        linears += f" (+{len(info['linears'])-4} more)"
    print(f"{marker} {idx:>4} | {info['type']:>10} | {info['class']:>30} | {linears}")

In [ ]:
# Create eval dataloader
eval_config = DistillationConfig(batch_size=2, seq_length=1024)
eval_loader = create_dataloader(eval_config, tokenizer)

# Measure baseline perplexity
print("Measuring baseline perplexity...")
baseline = compute_model_perplexity(model, eval_loader, num_samples=50, device=DEVICE)
print(f"\n📊 BASELINE: Loss = {baseline['loss']:.4f}, Perplexity = {baseline['perplexity']:.2f}")

---
## 🔬 Phase B: Surgical BitLinear Conversion

Convert the 18 DeltaNet layers from `nn.Linear` to `BitLinear` (ternary weights).
The 6 Full Attention layers stay at full precision.

In [ ]:
# Perform surgery
surgery_config = SurgeryConfig(aggression=SURGERY_AGGRESSION)
report = convert_model(model, surgery_config)

# Print report
print(surgical_report(model, report))

In [ ]:
# Measure post-surgery perplexity (expect significant degradation)
print("Measuring post-surgery perplexity (expect degradation)...")
post_surgery = compute_model_perplexity(model, eval_loader, num_samples=50, device=DEVICE)

print(f"\n📊 POST-SURGERY: Loss = {post_surgery['loss']:.4f}, Perplexity = {post_surgery['perplexity']:.2f}")
print(f"   Quality drop: {((post_surgery['perplexity'] / baseline['perplexity']) - 1) * 100:.1f}% perplexity increase")

---
## 🎓 Phase C: Knowledge Distillation

Train the BitLinear student to recover quality by distilling from the original
(full-precision) teacher model. The combined loss uses:
- Cross-entropy on hard labels (next-token prediction)
- KL divergence on temperature-scaled soft labels from the teacher
- Temperature annealing from T=4.0 → T=1.0 (cosine schedule)

In [ ]:
# Load teacher model (frozen, BF16 to save memory)
print(f"Loading teacher: {TEACHER_MODEL}...")
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).to(DEVICE)

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f"Teacher: {teacher_params:,} params ({teacher_params * 2 / 1e9:.2f} GB in BF16)")

# Check memory
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {allocated:.1f} / {total_mem:.1f} GB used")

In [ ]:
# Configure distillation
distill_config = DistillationConfig(
    num_steps=NUM_DISTILL_STEPS,
    checkpoint_dir=CHECKPOINT_DIR,
)
distill_config.auto_configure(gpu_vram)

# Create training dataloader
train_loader = create_dataloader(distill_config, tokenizer)

# Create trainer
trainer = DistillationTrainer(model, teacher, distill_config)

print(f"\nEffective batch: {distill_config.effective_batch_tokens:,} tokens/step")
print(f"Total tokens: ~{distill_config.effective_batch_tokens * distill_config.num_steps / 1e6:.0f}M")

In [ ]:
# Optional: resume from checkpoint
# trainer.load_checkpoint("/content/drive/MyDrive/recurrent_bitnet/checkpoints/checkpoint-2500.pt")

In [ ]:
# Run distillation!
trainer.train(train_loader)

In [ ]:
# Plot loss curves
trainer.plot_loss_curves()

In [ ]:
# Measure post-distillation perplexity
print("Measuring post-distillation perplexity...")
post_distill = compute_model_perplexity(model, eval_loader, num_samples=50, device=DEVICE)

print(f"\n📊 POST-DISTILLATION: Loss = {post_distill['loss']:.4f}, Perplexity = {post_distill['perplexity']:.2f}")

---
## 📈 Results: The Three-Number Story

In [ ]:
import matplotlib.pyplot as plt

# Summary comparison
phases = ["Baseline\n(Original)", "Post-Surgery\n(BitLinear, no training)", "Post-Distillation\n(BitLinear, trained)"]
ppls = [baseline["perplexity"], post_surgery["perplexity"], post_distill["perplexity"]]
losses = [baseline["loss"], post_surgery["loss"], post_distill["loss"]]
colors = ["#2ecc71", "#e74c3c", "#3498db"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Perplexity comparison
bars1 = ax1.bar(phases, ppls, color=colors, edgecolor="white", linewidth=2)
ax1.set_title("Perplexity Comparison", fontsize=14, fontweight="bold")
ax1.set_ylabel("Perplexity (lower is better)")
for bar, ppl in zip(bars1, ppls):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{ppl:.1f}", ha="center", fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

# Loss comparison
bars2 = ax2.bar(phases, losses, color=colors, edgecolor="white", linewidth=2)
ax2.set_title("Cross-Entropy Loss Comparison", fontsize=14, fontweight="bold")
ax2.set_ylabel("Loss (nats, lower is better)")
for bar, loss in zip(bars2, losses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{loss:.3f}", ha="center", fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("RecurrentBitNet V3: Selective BitLinear Validation", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print summary
stats = count_ternary_params(model)
recovery = 1.0 - (post_distill["perplexity"] - baseline["perplexity"]) / (post_surgery["perplexity"] - baseline["perplexity"])
print(f"\n{'='*60}")
print(f"  RESULTS SUMMARY")
print(f"{'='*60}")
print(f"  Baseline perplexity:       {baseline['perplexity']:.2f}")
print(f"  Post-surgery perplexity:   {post_surgery['perplexity']:.2f}  (+{((post_surgery['perplexity']/baseline['perplexity'])-1)*100:.1f}%)")
print(f"  Post-distillation:         {post_distill['perplexity']:.2f}  (+{((post_distill['perplexity']/baseline['perplexity'])-1)*100:.1f}%)")
print(f"  Quality recovery:          {recovery*100:.1f}%")
print(f"  BitLinear params:          {stats['bitlinear_pct']:.1f}% of model")
print(f"  Est. inference size:       {stats['est_size_mb']:.0f} MB")
print(f"{'='*60}")

---
## 💾 Save & Export

In [ ]:
# Save the final distilled model
save_path = os.path.join(CHECKPOINT_DIR, "recurrent_bitnet_v3_final")
os.makedirs(save_path, exist_ok=True)

# Save model + tokenizer
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save experiment metadata
import json
metadata = {
    "model": STUDENT_MODEL,
    "teacher": TEACHER_MODEL,
    "surgery_aggression": SURGERY_AGGRESSION,
    "distill_steps": trainer.global_step,
    "baseline_ppl": baseline["perplexity"],
    "post_surgery_ppl": post_surgery["perplexity"],
    "post_distill_ppl": post_distill["perplexity"],
    "param_stats": count_ternary_params(model),
}
with open(os.path.join(save_path, "experiment_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"✅ Model saved to {save_path}")
print(f"   Metadata: experiment_metadata.json")

---
## 🔮 What's Next: V2 Architecture

If this V3 validation shows that DeltaNet layers tolerate ternary quantization
(quality recovery > 80%), the next step is **V2**: weight-sharing recurrence
on top of the BitLinear'd Qwen3.5 layers.

V2 adds:
- **Depth recurrence**: same blocks execute R times with iteration embeddings
- **Adaptive halting**: stop recurring when representation stabilizes
- **Progressive recurrence curriculum**: train R=1→2→3→4
- **Auxiliary per-recurrence loss**: each intermediate step predicts next token

This would create a model with 298M unique params that behaves like 430M+,
running inference in ~75 MB.